***

# **Zillow Translations**

***

These are translations of all of the scripts that Seth made.

***

## **Packages Importing**

***

In [35]:
import pandas as pd 
import os
import numpy as np
import geopandas as gpd

***

## **Cost_1**

***

In [23]:
################################################### Cost_1a file ##########################################################################

# Classifier state
sacog_state = ["Sacramento, CA", "Yuba City, CA"]

ca_peers = ["Los Angeles, CA", "San Francisco, CA", "Riverside, CA", 
            "San Diego, CA", "Oxnard, CA", "Santa Rosa, CA", 
            "Vallejo, CA", "El Centro, CA", "Napa, CA"]

other_peers = ["Austin, TX", "Charlotte, NC", "Cincinnati, OH",
               "Cleveland, OH", "Columbus, OH", "Detroit, MI",
               "Indianapolis, IN", "Kansas City, KS", "Miami, FL",
               "Orlando, FL", "Phoenix, AZ", "Pittsburg, PA",
               "Portland, OR", "Salt Lake City, UT", "San Antonio, TX",
               "St. Louis, MO", "Tampa, FL"]

# More classifiers
sacog = ["Yuba City", "Sacramento"]
mtc = ["San Francisco", "Santa Rosa", "Vallejo", "Napa"]
scag = ["Los Angeles", "Riverside", "Oxnard", "El Centro"]

# Load CSV file
sale_price = pd.read_csv("Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")
sale_price = sale_price[sale_price['RegionType'] == 'msa']
sale_price = pd.melt(sale_price, id_vars=sale_price.columns[:5], var_name='Date', value_name='Price')

# Subset and rename cols
sale_price = sale_price[['RegionName', 'StateName', 'Date', 'Price']]
sale_price.columns = ['Region', 'State', 'Date', 'Price']
sale_price = sale_price[sale_price['Region'].isin(sacog_state + ca_peers + other_peers)]

# Removing trailing text from Region
sale_price['Region'] = sale_price['Region'].str.replace(r',.*', '', regex=True)

# Mutate function to get MPO col
sale_price['MPO'] = np.where(sale_price['Region'].isin(sacog), 'SACOG',
                             np.where(sale_price['Region'].isin(mtc), 'MTC',
                                      np.where(sale_price['Region'].isin(scag), 'SCAG',
                                               np.where(sale_price['Region'] == 'San Diego', 'SANDAG', sale_price['Region']))))

# Calculate median prices for each MPO
sacog_mn = sale_price[sale_price['MPO'] == 'SACOG'].groupby(['MPO', 'Date'])['Price'].median().reset_index()
sacog_mn['State'] = 'CA'
sacog_mn['Region'] = 'SACOG Median'
sacog_mn.columns = ['MPO', 'Date', 'Price', 'State', 'Region']

mtc_mn = sale_price[sale_price['MPO'] == 'MTC'].groupby(['MPO', 'Date'])['Price'].median().reset_index()
mtc_mn['State'] = 'CA'
mtc_mn['Region'] = 'MTC Median'
mtc_mn.columns = ['MPO', 'Date', 'Price', 'State', 'Region']

scag_mn = sale_price[sale_price['MPO'] == 'SCAG'].groupby(['MPO', 'Date'])['Price'].median().reset_index()
scag_mn['State'] = 'CA'
scag_mn['Region'] = 'SCAG Median'
scag_mn.columns = ['MPO', 'Date', 'Price', 'State', 'Region']

# Combine all median dfs, and clean cols
sale_price = pd.concat([sale_price, sacog_mn, mtc_mn, scag_mn], ignore_index=True)
sale_price = sale_price.sort_values(by=['MPO', 'Date', 'Region'])
sale_price = sale_price[['MPO', 'Region', 'State', 'Date', 'Price']]

display(sale_price.head(6))

In [168]:
################################################### Cost_1b file ##########################################################################

# Define SACOG counties
sacog_counties = ["El Dorado County", "Placer County", "Sacramento County",
                  "Sutter County", "Yolo County", "Yuba County"]

# Read the ZIP code shapefile
zip_codes = gpd.read_file("California_Zip_Codes.shp")

# Read the ZIP price data
zip_price = pd.read_csv("Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")

# # Filter zip_price for SACOG counties and California state
zip_price = zip_price[(zip_price['CountyName'].isin(sacog_counties)) & (zip_price['State'] == 'CA')]

# # Ensure ZIP_CODE column is integer for merging
zip_codes['ZIP_CODE'] = zip_codes['ZIP_CODE'].astype(int)
zip_price['RegionName'] = zip_price['RegionName'].astype(int)

# Merge the data with zip_codes
vals = zip_codes.merge(zip_price, left_on='ZIP_CODE', right_on='RegionName', how='inner')

# Sort by ZIP_CODE
vals = vals.sort_values('ZIP_CODE')

# # Select relevant columns
date_columns = [col for col in zip_price.columns if col.startswith('2')]
cols_to_select = ['ZIP_CODE', 'PO_NAME', 'City', 'CountyName'] + date_columns
vals = vals[cols_to_select]


# Melt the DataFrame
vals = pd.melt(vals, id_vars=['ZIP_CODE', 'PO_NAME', 'City', 'CountyName'], var_name='Date', value_name='Price')

# Rename columns
vals.columns = ['Zip', 'PO', 'City', 'County', 'Date', 'Price']

display(vals.head(6))

# Filter zip_codes for the ones present in vals
zip_codes = zip_codes[zip_codes['ZIP_CODE'].astype(int).isin(vals['Zip'])]
display(zip_codes.head(6))

,Zip,PO,City,County,Date,Price
0,95602,Auburn,Auburn,Placer County,2000-01-31,213290.006754
1,95603,Auburn,Auburn,Placer County,2000-01-31,212200.401185
2,95605,West Sacramento,West Sacramento,Yolo County,2000-01-31,89855.548858
3,95607,Capay,Capay,Yolo County,2000-01-31,NaN
4,95608,Carmichael,Carmichael,Sacramento County,2000-01-31,166106.721610
5,95610,Citrus Heights,Citrus Heights,Sacramento County,2000-01-31,136912.800306


,OBJECTID,ZIP_CODE,PO_NAME,STATE,POPULATION,POP_SQMI,SQMI,SHAPE_Leng,SHAPE_Area,geometry
1427,1428,95602,Auburn,CA,18464,347.07,53.20,257364.480106,1.483137e+09,"POLYGON ((-292287.956 9496967.353, -292378.01 ..."
1428,1429,95603,Auburn,CA,28358,723.42,39.20,320937.670841,1.092931e+09,"POLYGON ((-271292.246 9480517.245, -271595.151..."
1429,1430,95605,West Sacramento,CA,15029,3638.98,4.13,55484.000496,1.152905e+08,"POLYGON ((-430590.519 9338435.686, -430265.421..."
1431,1432,95607,Capay,CA,473,8.24,57.42,278341.093476,1.600608e+09,"POLYGON ((-589217.499 9360878.291, -589005.42 ..."
1432,1433,95608,Carmichael,CA,61115,4679.56,13.06,113153.948913,3.640989e+08,"POLYGON ((-374994.385 9364455.216, -374681.852..."
...,...,...,...,...,...,...,...,...,...,...
1714,1715,96143,Kings Beach,CA,4086,225.50,18.12,114901.065469,5.052135e+08,"POLYGON ((-1504.03 9597407.396, -1530.662 9596..."
1715,1716,96145,Tahoe City,CA,3389,21.66,156.46,497266.189360,4.361518e+09,"POLYGON ((-11169.546 9568933.846, -10885.192 9..."
1716,1717,96146,Olympic Valley,CA,1152,149.42,7.71,76824.251287,2.151139e+08,"POLYGON ((-57050.818 9560214.046, -56237.8 955..."
1717,1718,96148,Tahoe Vista,CA,1487,991.33,1.50,32123.269836,4.177740e+07,"POLYGON ((-11169.546 9568933.846, -11563.127 9..."


***

## **Cost 2**

***

In [157]:
################################################### Cost_2a file ##########################################################################

# Read in data
rent_price = pd.read_csv("Metro_zori_uc_sfrcondomfr_sm_month.csv") #nolint
rent_price = rent_price[rent_price['RegionType'] == "msa"]

# Melt data down
rent_price = pd.melt(rent_price, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'])

# Keep important cols and rename
rent_price = rent_price[['RegionName', 'StateName', 'variable', 'value']]
rent_price.columns = ["Region", "State", "Date", "Price"]

# Subset data 
rent_price = rent_price[rent_price['Region'].isin(sacog_state + ca_peers + other_peers)]

# Mutate function to make the MPO col
rent_price['MPO'] = np.where(rent_price['Region'].str.contains('|'.join(sacog)), 'SACOG',
                             np.where(rent_price['Region'].str.contains('|'.join(mtc)), 'MTC',
                                      np.where(rent_price['Region'].str.contains('|'.join(scag)), 'SCAG',
                                               np.where(rent_price['Region'] == 'San Diego', 'SANDAG', rent_price['Region']))))

# This is similar to before. We want to get the median price and subset accordingly based on MPO.
sacog_mn = rent_price[rent_price['MPO'] == 'SACOG'].groupby(['MPO', 'Date'])['Price'].median().reset_index()
sacog_mn['State'] = 'CA'
sacog_mn['Region'] = 'SACOG Median'
sacog_mn.columns = ['MPO', 'Date', 'Price', 'State', 'Region']

mtc_mn = rent_price[rent_price['MPO'] == 'MTC'].groupby(['MPO', 'Date'])['Price'].median().reset_index()
mtc_mn['State'] = 'CA'
mtc_mn['Region'] = 'MTC Median'
mtc_mn.columns = ['MPO', 'Date', 'Price', 'State', 'Region']

scag_mn = rent_price[rent_price['MPO'] == 'SCAG'].groupby(['MPO', 'Date'])['Price'].median().reset_index()
scag_mn['State'] = 'CA'
scag_mn['Region'] = 'SCAG Median'
scag_mn.columns = ['MPO', 'Date', 'Price', 'State', 'Region']

# Concatenate data
rent_price = pd.concat([rent_price, sacog_mn, mtc_mn, scag_mn], ignore_index=True)

# Sort data
rent_price = rent_price.sort_values(by=['MPO', 'Date', 'Region'])

# Reorder cols
rent_price = rent_price[['MPO', 'Region', 'State', 'Date', 'Price']]
display(rent_price.head(6))

,MPO,Region,State,Date,Price
15,"Austin, TX","Austin, TX",TX,2015-01-31,1179.638820
41,"Austin, TX","Austin, TX",TX,2015-02-28,1191.871533
67,"Austin, TX","Austin, TX",TX,2015-03-31,1198.160400
93,"Austin, TX","Austin, TX",TX,2015-04-30,1203.510386
119,"Austin, TX","Austin, TX",TX,2015-05-31,1214.436334
145,"Austin, TX","Austin, TX",TX,2015-06-30,1223.725306


In [158]:
# ################################################### Cost_2b file ##########################################################################

# Read files in
zip_codes = gpd.read_file("California_Zip_Codes.shp")
zip_rent = pd.read_csv("Zip_zori_uc_sfrcondomfr_sm_month.csv")

# Filter the zip_rent df for SACOG counties in CA
zip_rent = zip_rent[(zip_rent['CountyName'].isin(sacog_counties)) & (zip_rent['State'] == "CA")]

# Convert zip code to integer in both the dfs
zip_codes['ZIP_CODE'] = zip_codes['ZIP_CODE'].astype(int)
zip_rent['RegionName'] = zip_rent['RegionName'].astype(int)

# Merge the zip codes df with the rent df. This will end up being what we use for the final frame. 
vals = zip_codes.merge(zip_rent, left_on='ZIP_CODE', right_on='RegionName')

# Arrange by zip
vals = vals.sort_values(by='ZIP_CODE')

# Take all of the date columns for meting later. We then subset VALS to only include info we want. 
date_columns = [col for col in zip_rent.columns if col.startswith('2')]
vals = vals[['ZIP_CODE', 'PO_NAME', 'City', 'CountyName'] + date_columns]

# Melt the dataframe
vals = pd.melt(vals, id_vars=['ZIP_CODE', 'PO_NAME', 'City', 'CountyName'], var_name='Date', value_name='Rent')

display(vals.head(6))


,ZIP_CODE,PO_NAME,City,CountyName,Date,Rent
0,95603,Auburn,Auburn,Placer County,2015-01-31,NaN
1,95605,West Sacramento,West Sacramento,Yolo County,2015-01-31,NaN
2,95608,Carmichael,Carmichael,Sacramento County,2015-01-31,NaN
3,95610,Citrus Heights,Citrus Heights,Sacramento County,2015-01-31,NaN
4,95616,Davis,Davis,Yolo County,2015-01-31,NaN
5,95618,Davis,Davis,Yolo County,2015-01-31,NaN
